In [2]:
# imports

import json
import os
from dotenv import load_dotenv
from openai import OpenAI

import anthropic
from IPython.display import Markdown, display, update_display
import google.generativeai


In [4]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exists")
else:
    print("OpenAI API Key not set")
    


OpenAI API Key exists


In [5]:
# Connect to OpenAI

openai = OpenAI()


In [6]:
JUDGE_SYSTEM_PROMPT = """You are an expert clinical educator evaluating simulated emergency department conversations.

You MUST respond with valid JSON only.
Do not include markdown, explanations, or extra text.
Do not wrap the JSON in code fences.

Evaluate realism, role fidelity, and educational suitability.
Be conservative in your ratings.
"""


In [7]:
JUDGE_USER_PROMPT = """
Conversation:
----------------
{FULL_CONVERSATION_TEXT}
----------------

Please rate the conversation using the following scale:

1 = Strongly disagree
2 = Disagree
3 = Neutral
4 = Agree
5 = Strongly agree

Return your answer in JSON with EXACTLY this structure:

{{
  "role_fidelity": 1-5,
  "turn_coherence": 1-5,
  "communication_realism": 1-5,
  "educational_usable": true/false,
  "comments": "string"
}}
"""



In [8]:
def evaluate_conversation_automatically(conversation_text, model="gpt-4o"):
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": JUDGE_USER_PROMPT.format(
                FULL_CONVERSATION_TEXT=conversation_text
            )
        }
    ]

    response = openai.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )

    return json.loads(response.choices[0].message.content)



In [9]:
## only for a single file 

conversations = []

with open("simulations.jsonl", "r", encoding="utf-8") as fh:
    for line in fh:
        conversations.append(json.loads(line))



In [10]:
## only for a single file 

with open("data/conversations/conv_73cf2ae2.json", "r", encoding="utf-8") as fh:
    convo = json.load(fh)


In [11]:
# only for  a single file  
conversation_text = "\n".join(
    f"{t['speaker'].upper()}: {t['text']}"
    for t in convo["turns"]
)



In [12]:
conversation_text = conversation_to_text(convo)
print(conversation_text[:1000])  # sanity check


NameError: name 'conversation_to_text' is not defined

In [50]:
judge_result = evaluate_conversation_automatically(conversation_text)
print(judge_result)



{'role_fidelity': 4, 'turn_coherence': 4, 'communication_realism': 4, 'educational_usable': True, 'comments': 'The conversation demonstrates good role fidelity, with the doctor and nurse performing their roles appropriately, though there was a minor role violation with the patient initially. The turn coherence is strong, with logical progression and responses that align with the scenario. Communication realism is high, capturing the urgency and emotional weight of a potential heart attack situation. The scenario is educationally useful for demonstrating effective communication and management in an emergency setting, though the role violation could be addressed for improved fidelity.'}


In [ ]:
############################# batch files ########################

In [8]:
def conversation_json_to_text(convo_json):
    """
    Converts a saved conversation JSON into a readable transcript
    suitable for LLM judging.
    """
    lines = []
    for turn in convo_json["turns"]:
        speaker = turn["speaker"].upper()
        text = turn["text"].strip()
        lines.append(f"{speaker}: {text}")
    return "\n\n".join(lines)


In [9]:
from pathlib import Path

def evaluate_conversation_batch(
    conversations_dir,
    output_path="data/judge_results.jsonl",
    model="gpt-4o"
):
    """
    Evaluates all conversation JSON files in a directory using the LLM judge.

    Parameters
    ----------
    conversations_dir : str or Path
        Folder containing conversation JSON files
    output_path : str
        Where to append judge results (JSONL)
    model : str
        LLM model to use as judge
    """

    conversations_dir = Path(conversations_dir)
    results = []

    for json_file in conversations_dir.glob("*.json"):
        with open(json_file, "r", encoding="utf-8") as fh:
            convo = json.load(fh)

        conversation_text = conversation_json_to_text(convo)

        try:
            judge_result = evaluate_conversation_automatically(
                conversation_text,
                model=model
            )

            record = {
                "conversation_id": convo["conversation_id"],
                "file": json_file.name,
                "num_turns": convo["num_turns"],
                "role_guard_failures": convo["role_guard_failures"],
                "judge": judge_result
            }

            results.append(record)

            # append immediately (safe for long runs)
            with open(output_path, "a", encoding="utf-8") as out:
                out.write(json.dumps(record, ensure_ascii=False) + "\n")

            print(f"✓ Evaluated {json_file.name}")

        except Exception as e:
            print(f"✗ Failed on {json_file.name}: {e}")

    return results


In [12]:
results = evaluate_conversation_batch(
    conversations_dir="data/conversations",
    output_path="data/judge_results.jsonl",
    model="gpt-4o"
)


✓ Evaluated conv_02263e1d.json
✓ Evaluated conv_07aef0b9.json
✓ Evaluated conv_0e3ec2e6.json
✓ Evaluated conv_0f1141ee.json
✓ Evaluated conv_10a6c7dc.json
✓ Evaluated conv_2bf529b2.json
✓ Evaluated conv_30300709.json
✓ Evaluated conv_3c5a1b05.json
✓ Evaluated conv_46b0e3b2.json
✓ Evaluated conv_46defe37.json
✓ Evaluated conv_4a73a2e8.json
✓ Evaluated conv_544e2322.json
✓ Evaluated conv_568cb6e9.json
✓ Evaluated conv_5c0c9b3f.json
✓ Evaluated conv_5e0b4438.json
✓ Evaluated conv_73cf2ae2.json
✓ Evaluated conv_7b0158e1.json
✓ Evaluated conv_7b2a03ea.json
✓ Evaluated conv_81c04993.json
✓ Evaluated conv_88881828.json
✓ Evaluated conv_990e2cac.json
✓ Evaluated conv_aadab8dc.json
✓ Evaluated conv_af2283d0.json
✓ Evaluated conv_b306aeb3.json
✓ Evaluated conv_be0a4281.json
✓ Evaluated conv_c10efbc5.json
✓ Evaluated conv_e101cfd4.json
✓ Evaluated conv_f6002de1.json
✓ Evaluated conv_f98d056d.json
